# Practical 1

## *fMRIPrep* setup

Install the dataset with datalad. This downloads all file metadata, but not the file contents.

In [ ]:
%%bash
datalad --log-level "warning" \
    install "https://github.com/OpenNeuroDatasets/ds000117.git"

Download the file contents for a single scan as well as the associated anatomical and field map scans. This gives us a minimal set of files for running *fMRIPrep*.

Running `cd` inside the cell to change the current working directory is required for `datalad` to know which dataset it's working on. In contrast to a bash script, such changes in `%%bash` cells are temporary and only affect the current cell.

In [ ]:
%%bash
cd "ds000117"
datalad get sub-01/ses-mri/{anat,fmap,func/sub-01_ses-mri_task-facerecognition_run-01_bold.nii.gz}

*fMRIPrep* uses FreeSurfer for several steps, which needs a license file. You can get your free license by filling in the [registration form](https://surfer.nmr.mgh.harvard.edu/registration.html) or use the following cell to write a temporary one just this once.

In [ ]:
%%bash
echo "b2VzdGViYW5Ac3RhbmZvcmQuZWR1CjMwNzU2CiAqQ1MzYkJ5VXMxdTVNCiBGU2kvUGJsejJxR1V3Cg==" | 
    base64 --decode > "license.txt"

Workflows such as *fMRIPrep* are made available via `module` in the NeuroDesk environment.

In [ ]:
import module
import os

await module.load('fmriprep/25.2.5')

We now have the `fmriprep` command available in this notebook. The command has several options.

In [ ]:
%%bash
fmriprep --help

As you can see, *fMRIPrep* has quite a few command line flags! Let's go through them together.

All BIDS Apps have three mandatory command line flags. These are
1. Path to the input BIDS dataset
2. Path to put output files
3. The analysis level. For *fMRIPrep* that is always `participant`, meaning that each subject is processed independently, but for other tools levels such as `session` or `group` or may also be implemented.

All of the other command line flags are optional. My favorite optional ones are

| Flag                            | Description                                                      |
| :------------------------------ | :--------------------------------------------------------------- |
| `--participant-label`           | Select one or more participants for processing.                  |
| `--skip-bids-validation`        | Datalad datasets where only some files were downloaded always fail BIDS validation. This flag prevents the error by skipping the validation step. |
| `--boilerplate-only`            | Runs only the setup stage of *fMRIPrep* so that you can inspect the log and make sure that all your files have been recognized correctly. |
| `--fs-no-reconall`              | Disables surface reconstruction. If you know you're not going to use the surface data, then this can save you several hours of processing time. |
| `--fs-license-file`             | Even with surface reconstruction disabled, *fMRIPrep* will always run a few command from FreeSurfer, meaning that a license is required. This flag makes it easy to pass in your license file. |
| `--dummy-scans`                 | If your scanner does not automatically discard the first few volumes of your scan, you can add the number of volumes to ignore here. Note that *fMRIPrep* still includes these volumes in the output file, so they will also need to be ignored by subsequent analyses. |
| `--nprocs` and `--omp-nthreads` | How many cores to use. |
| `--mem`                         | *fMRIPrep* consists of many commands that need to be run, and the developers have added a (mostly accurate) prediction for how much memory each will use. *fMRIPrep* will not run more commands simultaneously if their combined memory usage prediction will exceed the memory limit specified here. If you are experiencing out-of-memory crashes, setting this to a low-ish number such as $10,000$ can help. |

Subject 1 has multiple scans, but we only downloaded the first one. A filter file can be passed as an optional input to *fMRIPrep* to specify that we only want to process that one. The following cell creates the filter file using heredoc syntax.

In [ ]:
%%bash
cat <<JSON > "filter.json"
{
    "bold": {
        "run": "1"
    }
}
JSON

The next cell runs in `--boilerplate-only` mode, so that we can see that all stages are setup correctly. As ds000117 is not a multi-band dataset, we can safely ignore the message about the missing single-band reference file.

In [ ]:
%%bash
unset APPTAINERENV_SUBJECTS_DIR
fmriprep \
    "ds000117" "ds000117-fmriprep" "participant" \
    --participant-label "01" \
    --bids-filter-file "filter.json" \
    --fs-license-file "license.txt" \
    --skip-bids-validation \
    --boilerplate-only

Compare the outputs after we delete the field maps

In [ ]:
%%bash
rm --recursive --force "ds000117/sub-01/ses-mri/fmap"

In [ ]:
%%bash
unset APPTAINERENV_SUBJECTS_DIR
fmriprep \
    "ds000117" "ds000117-fmriprep" "participant" \
    --participant-label "01" \
    --bids-filter-file "filter.json" \
    --fs-license-file "license.txt" \
    --skip-bids-validation \
    --boilerplate-only

We can restore the field maps with `git`

In [ ]:
%%bash
cd "ds000117"
git reset --hard HEAD

And run the actual preprocessing by removing the `--boilerplate-only` command line flag. This will take some time

In [ ]:
%%bash
unset APPTAINERENV_SUBJECTS_DIR
fmriprep \
    "ds000117" "ds000117-fmriprep" "participant" \
    --participant-label "01" \
    --bids-filter-file "filter.json" \
    --fs-license-file "license.txt" \
    --skip-bids-validation

### Bonus

What happens if you remove the `anat` scans before running *fMRIPrep*?